# Lab 7 — Project: Logistic Regression

Binary Logistic Regression analysis on the project dataset.

**Primary application:** $\pi_T = 0.1$, $C_{fn} = C_{fp} = 1$

**Sections:**
1. Standard LR — full dataset, varying λ
2. Standard LR — reduced dataset (1/50), studying overfitting
3. Prior-weighted LR — adapting the model to the target prior
4. Quadratic LR — nonlinear feature expansion
5. Final comparison — all models including Gaussian classifiers

In [ ]:
import numpy as np
import scipy.optimize
import scipy.special
import matplotlib.pyplot as plt
import pandas as pd

## Helper functions — Bayes risk metrics

In [ ]:
def vcol(x): 
    """
        Convert Data to Column Vector  
    """
    return x.reshape((x.size, 1))

def vrow(x): 
    """
        Convert Data to Row Vector
    """
    return x.reshape((1, x.size))

In [ ]:
def compute_confusion_matrix(pred, labels):

    """
    Assume that classes are labeled 0, 1, 2 ... (nClasses - 1)

        - nClasses: Number of Classes
        - M: Confusion Matrix

    Rows = Predictions
    Columns = True Classes
    """
    nC = labels.max() + 1
    M = np.zeros((nC, nC), dtype=np.int32)
    for i in range(labels.size): M[pred[i], labels[i]] += 1
    return M

`computer_optimal_Bayes_binary_llr`:

In a binary task (where hypotheses are True $\mathcal{H}_T=1$ and False $\mathcal{H}_F=0$), we have two specific costs: $C_{fn}$ (the cost of a false negative) and $C_{fp}$ (the cost of a false positive). The document mathematically proves that to minimize your expected risk, you can consolidate your prior beliefs ($\pi_1$) and your costs into a single decision threshold $t$:

$$t = -\log\frac{\pi_1 C_{fn}}{(1-\pi_1)C_{fp}}$$

In [ ]:
# Doing the classification with the 
    #  t   --> threshold 
    #  llr --> log-likelihood ratio

def compute_optimal_Bayes_binary_llr(llr, prior, Cfn, Cfp):
    """
        Optimal Bayes decision for binary task with LLR scores.
        Compares LLR against threshold t = -log(pi1*Cfn / (1-pi1)*Cfp).
        Predict class 1 if LLR > t, else class 0.
    """
    return np.int32(llr > -np.log(prior * Cfn / ((1 - prior) * Cfp)))

- $P_{fn}$ **(False Negative Rate)**:

    It takes the number of False Negatives ($M_{0,1}$, which is Predicted 0, True 1) and divides it by the total number of actual Class 1 samples ($M_{0,1} + M_{1,1}$)


- $P_{fp}$ **(False Positive Rate)**:

    It takes the number of False Positives ($M_{1,0}$, which is Predicted 1, True 0) and divides it by the total number of actual Class 0 samples ($M_{0,0} + M_{1,0}$). 


**The Math:** For a binary task, the expected risk ($\mathcal{B}$ or $DCF_u$) is simply the sum of your two possible errors, weighted by their respective costs and prior probabilities.
  $$DCF_u = \pi_1 C_{fn} P_{fn} + (1-\pi_1) C_{fp} P_{fp}$$

  $$\mathcal{B}_{dummy} = \min(\pi_1 C_{fn}, (1-\pi_1) C_{fp})$$

  $$DCF_{normalized} = \frac{DCF_u}{\mathcal{B}_{dummy}} 

In [ ]:
def compute_empirical_Bayes_risk_binary(pred, labels, prior, Cfn, Cfp, normalize=True):
    """
        Optimal Bayes decision for binary task with LLR scores.
        Compares LLR against threshold t = -log(pi1*Cfn / (1-pi1)*Cfp).
        Predict class 1 if LLR > t, else class 0.
    """
    M = compute_confusion_matrix(pred, labels)
    Pfn = M[0,1] / (M[0,1] + M[1,1])
    Pfp = M[1,0] / (M[0,0] + M[1,0])
    dcf = prior * Cfn * Pfn + (1 - prior) * Cfp * Pfp
    return dcf / np.minimum(prior * Cfn, (1 - prior) * Cfp) if normalize else dcf

`compute_Pfn_Pfp_fast`

Your model just gave you a bunch of raw scores (LLRs) for your test data. To really know how good your model is, you can't just test one single decision boundary. You need to know how the model behaves if you shift the boundary from "accept everyone" all the way to "reject everyone."

This function is a mass-evaluation machine. It systematically tests every single meaningful threshold possible for the dataset and calculates your error rates at each one.

**Returns**

- `thOut` (**Thresholds**): The specific decision boundary being tested at that moment.

- `PfnOut` (**False Negative Rate**): At this specific threshold, what percentage of actual Class 1s did the model accidentally call Class 0?

- `PfOut` (**False Positive Rate**): At this specific threshold, what percentage of actual Class 0s did the model accidentally call Class 1?

In [ ]:
def compute_Pfn_Pfp_fast(llr, labels):
    """
    Compute Pfn and Pfp

    - P_fn

    - P_fp    
    
    """
    sorter = np.argsort(llr)
    ls, cs = llr[sorter], labels[sorter]
    nT, nF = (cs==1).sum(), (cs==0).sum()
    nFN, nFP = 0, nF
    Pfn, Pfp = [nFN/nT], [nFP/nF]
    for i in range(len(ls)):
        if cs[i] == 1: nFN += 1
        else: nFP -= 1
        Pfn.append(nFN/nT); Pfp.append(nFP/nF)
    ls2 = np.concatenate([[-np.inf], ls])
    PfnO, PfpO, thO = [], [], []
    for i in range(len(ls2)):
        if i == len(ls2)-1 or ls2[i+1] != ls2[i]:
            PfnO.append(Pfn[i]); PfpO.append(Pfp[i]); thO.append(ls2[i])
    return np.array(PfnO), np.array(PfpO), np.array(thO)


- `compute_Pfn_Pfp_allThresholds_fast` **is the engine:** 
    
    It does all the hard work. It sweeps the line across the sorted scores and spits out the raw error rates ($P_{fn}$ and $P_{fp}$) for every single possible threshold.

- `compute_minDCF_binary_fast` **is just the calculator:** 

    It takes those massive lists of error rates, plugs them into the cost formula (which creates a massive list of DCF penalties), and then just looks at that list and says, "Which one of these numbers is the smallest?"

In [ ]:
def compute_minDCF(llr, labels, prior, Cfn, Cfp):
    Pfn, Pfp, _ = compute_Pfn_Pfp_fast(llr, labels)
    return ((prior*Cfn*Pfn + (1-prior)*Cfp*Pfp) / np.minimum(prior*Cfn, (1-prior)*Cfp)).min()



def compute_actDCF(llr, labels, prior, Cfn, Cfp):
    pred = compute_optimal_Bayes_binary_llr(llr, prior, Cfn, Cfp)
    return compute_empirical_Bayes_risk_binary(pred, labels, prior, Cfn, Cfp)

## LR model trainers

### Objective function (formula 2 from the lab)

$$J(\mathbf{w}, b) = \frac{\lambda}{2}\|\mathbf{w}\|^2 + \frac{1}{n}\sum_{i=1}^n \log\left(1 + e^{-z_i(\mathbf{w}^T\mathbf{x}_i + b)}\right), \quad z_i = 2c_i - 1$$

Gradient (provided analytically for speed — avoids expensive finite-difference approximation):

$$\nabla_{\mathbf{w}} J = \lambda \mathbf{w} + \frac{1}{n}\sum_i G_i \mathbf{x}_i, \quad \frac{\partial J}{\partial b} = \frac{1}{n}\sum_i G_i, \quad G_i = \frac{-z_i}{1 + e^{z_i(\mathbf{w}^T\mathbf{x}_i+b)}}$$

### Score → LLR conversion

LR scores are log-posterior-ratios under $\pi_{emp}$. To use Bayes thresholds, subtract the training prior log-odds:

$$s_{LLR} = s(\mathbf{x}) - \log\frac{\pi_{emp}}{1 - \pi_{emp}}$$

For the prior-weighted model, use $\pi_T$ instead of $\pi_{emp}$.

In [ ]:
def trainLogRegBinary(DTR, LTR, l):
    """
    Standard binary LR with L2 regularization.
    Computes objective + gradient analytically.
    Returns (w, b).

    - DTR: Training Data
    - LTR: Label Training Data
    - l: lambda - regularization hyper-parameter


    - What it does:
      It treats every single piece of training data exactly the same.
      If your training data is 90% dogs and 10% cats,
      the model naturally assumes the real world is also 90% dogs and 10% cats.
    """
    # Transform labels to -1 & 1
    ZTR = LTR * 2.0 - 1.0

    # Calculates loss & gradient
    def obj(v):

        # Params Unpacking
        w, b = v[:-1], v[-1]
        
        # Calculate the linear score 
        s = (vrow(w) @ DTR + b).ravel()

        # Calculate the loss
        loss = np.logaddexp(0, -ZTR * s)

        # Calculating the Gradients
        G = -ZTR / (1.0 + np.exp(ZTR * s))
        GW = (vrow(G) * DTR).mean(1) + l * w
        Gb = G.mean()

        return loss.mean() + l/2 * np.linalg.norm(w)**2, np.hstack([GW, Gb])
    
    # The Optimizer L-BFGS
    vf = scipy.optimize.fmin_l_bfgs_b(obj, x0=np.zeros(DTR.shape[0]+1))[0]
    return vf[:-1], vf[-1]

```python
s = (vrow(w) @ DTR + b).ravel()
```

$s(x_{i}) = w^{T}x_{i} + b$

<br>

```python
loss = np.logaddexp(0, -ZTR * s)
```

$\log(1 + e^{-z_{i}(w^{T}x_{i}+b)})$

In [ ]:
def trainWeightedLogRegBinary(DTR, LTR, l, pT):


    """
    Prior-weighted LR. Sample weights:
      xi = pT / nT  for class-1 samples
      xi = (1-pT) / nF  for class-0 samples
    Simulates training with prior pT.
    Score-to-LLR: s_llr = s - log(pT / (1-pT)).
    Returns (w, b).

    - DTR: Training Data
    - LTR: Training Data Label
    - pT:  Primary Application (effective prior)

    - what it does:

        It artificially rigs the learning process to simulate
        a completely different reality.  

        You give it a target probability (pT).
        By doing this, you are telling the model:

            "I don't care how many dogs and cats are actually in my training data,
            I want you to learn as if the real world is exactly pT percent dogs."


    """
    ZTR = LTR * 2.0 - 1.0
    wT = pT / (ZTR > 0).sum()
    wF = (1 - pT) / (ZTR < 0).sum()
    def obj(v):
        w, b = v[:-1], v[-1]
        s = (vrow(w) @ DTR + b).ravel()
        loss = np.logaddexp(0, -ZTR * s)
        loss[ZTR > 0] *= wT
        loss[ZTR < 0] *= wF
        G = -ZTR / (1.0 + np.exp(ZTR * s))
        G[ZTR > 0] *= wT
        G[ZTR < 0] *= wF
        GW = (vrow(G) * DTR).sum(1) + l * w
        Gb = G.sum()
        return loss.sum() + l/2 * np.linalg.norm(w)**2, np.hstack([GW, Gb])
    vf = scipy.optimize.fmin_l_bfgs_b(obj, x0=np.zeros(DTR.shape[0]+1))[0]
    return vf[:-1], vf[-1]

In [ ]:
def quadratic_expand(D):
    """
    Quadratic feature expansion: x ∈ R^d  →  φ(x) = [vec(x xᵀ), x] ∈ R^(d²+d)
    For d=6: output dim = 36 + 6 = 42
    Enables learning quadratic (elliptical) decision boundaries.

    - What it does:

        it is mapping the data to a higher dimensionality,
        so when apply the Linear Regression in those dimensions
        when we map it back to the original dimensions, it
        will be a non-linear decision boundry
    """
    n, d = D.shape[1], D.shape[0]
    out = np.zeros((d*d + d, n))
    for i in range(n):
        x = D[:, i]
        out[:, i] = np.concatenate([np.outer(x, x).ravel(), x])
    return out

## Load and split the project data

In [ ]:
def load_project_data(path):
    D, L = [], []
    with open(path) as f:
        for line in f:
            parts = [p.strip() for p in line.strip().split(',')]
            D.append([float(x) for x in parts[:-1]])
            L.append(int(parts[-1]))
    return np.array(D).T, np.array(L, dtype=np.int32)

def split_db(D, L, seed=0, ratio=2/3):
    nTrain = int(D.shape[1] * ratio)
    np.random.seed(seed)
    idx = np.random.permutation(D.shape[1])
    return (D[:, idx[:nTrain]], L[idx[:nTrain]]), (D[:, idx[nTrain:]], L[idx[nTrain:]])

D, L = load_project_data('../../../Project/trainData.txt')
(DTR, LTR), (DVAL, LVAL) = split_db(D, L)
pT   = 0.1
lambdas = np.logspace(-4, 2, 13)
pEmp = (LTR == 1).sum() / LTR.size
print(f'Training: {DTR.shape[1]} samples | Validation: {DVAL.shape[1]} samples')
print(f'Empirical prior: {pEmp:.4f}')

## Part 1: Standard LR — Full Dataset

Train for 13 log-spaced values of λ ∈ [$10^{-4}$, $10^2$].
Compute actDCF and minDCF for the primary application $\pi_T = 0.1$.

**Note:** We have to remove the empirical prior log-odds from the score before computing actDCF:

$$s_{LLR} = s(\mathbf{x}) - \log\frac{\pi_{emp}}{1 - \pi_{emp}}$$

In [ ]:
res_std = []
for lam in lambdas:
    w, b = trainLogRegBinary(DTR, LTR, lam)
    sVal = (vrow(w) @ DVAL + b).ravel()
    sLLR = sVal - np.log(pEmp / (1 - pEmp))
    act = compute_actDCF(sLLR, LVAL, pT, 1.0, 1.0)
    mn  = compute_minDCF(sLLR, LVAL, pT, 1.0, 1.0)
    res_std.append((lam, act, mn, sLLR))

display(pd.DataFrame([(f'{r[0]:.1e}', f'{r[1]:.4f}', f'{r[2]:.4f}') for r in res_std],
                     columns=['λ', 'actDCF', 'minDCF']))

In [ ]:
fig, ax = plt.subplots(figsize=(9,5))
ax.plot(lambdas, [r[1] for r in res_std], 'r-o', label='actDCF', linewidth=2)
ax.plot(lambdas, [r[2] for r in res_std], 'b--o', label='minDCF', linewidth=2)
ax.set_xscale('log'); ax.set_xlabel('λ (log scale)'); ax.set_ylabel('DCF')
ax.set_title('Standard LR — Full Dataset (πT = 0.1)')
ax.legend(); ax.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

### Answer: What do you observe?

**minDCF is flat (~0.361–0.365) across all λ.** With 4000 training samples, regularization has almost no effect on the discriminative ability of the model. The dataset is large enough that the learned boundary is already well-determined regardless of λ.

**actDCF, however, degrades severely as λ increases** — from ~0.40 at low λ to 1.0 at high λ. This is a **calibration problem**: heavy regularization shrinks the weights toward zero, compressing the score range. The threshold $t = -\log(\pi_T/(1-\pi_T)) = \log(9) \approx 2.20$ is no longer within the score range of the compressed model, so all samples get assigned to one class.

**Conclusion:** For large datasets, use small λ (e.g. $10^{-4}$). Regularization is not needed and actively hurts calibration.

## Part 2: Standard LR — Reduced Dataset (1 out of 50)

Keep only 1 sample in 50 from the training set (80 samples). Apply filter **after** splitting: `DTR[:, ::50]`, `LTR[::50]`. Validation set is unchanged.

In [ ]:
DTR_red = DTR[:, ::50]
LTR_red = LTR[::50]
pEmp_red = (LTR_red == 1).sum() / LTR_red.size
print(f'Reduced training set: {DTR_red.shape[1]} samples | pEmp_red = {pEmp_red:.4f}')

res_red = []
for lam in lambdas:
    w, b = trainLogRegBinary(DTR_red, LTR_red, lam)
    sVal = (vrow(w) @ DVAL + b).ravel()
    sLLR = sVal - np.log(pEmp_red / (1 - pEmp_red))
    act = compute_actDCF(sLLR, LVAL, pT, 1.0, 1.0)
    mn  = compute_minDCF(sLLR, LVAL, pT, 1.0, 1.0)
    res_red.append((lam, act, mn))

display(pd.DataFrame([(f'{r[0]:.1e}', f'{r[1]:.4f}', f'{r[2]:.4f}') for r in res_red],
                     columns=['λ', 'actDCF', 'minDCF']))

In [ ]:
fig, ax = plt.subplots(figsize=(9,5))
ax.plot(lambdas, [r[1] for r in res_red], 'r-o', label='actDCF', linewidth=2)
ax.plot(lambdas, [r[2] for r in res_red], 'b--o', label='minDCF', linewidth=2)
ax.set_xscale('log'); ax.set_xlabel('λ (log scale)'); ax.set_ylabel('DCF')
ax.set_title('Standard LR — Reduced Dataset 1/50 (πT = 0.1)')
ax.legend(); ax.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

### Answer: What do you observe? Can you explain the results?

With only 80 samples, regularization plays a decisive role — the bias-variance tradeoff is visible:

- **Low λ ($10^{-4}$):** actDCF ≈ 0.98 (near random). The model overfits the 80 training samples. Weights grow large, the decision boundary is highly specific to the training noise and does not generalize.
- **Optimal λ (~$10^{-2}$):** actDCF ≈ 0.47, minDCF ≈ 0.44. Best trade-off. Regularization constrains the weights enough to generalize, without over-compressing the scores.
- **High λ (~$1.0$+):** actDCF → 1.0. The model underfits — weights shrink to near-zero, all scores collapse, discriminative power is lost.

**Key insight:** With limited data, λ must be tuned. The optimal λ is significantly larger than for the full dataset. This confirms that regularization is not about the model complexity alone, but about the **ratio of model complexity to available data**.

## Part 3: Prior-Weighted LR (Full Dataset, πT = 0.1)

Objective:

$$J(\mathbf{w}, b) = \frac{\lambda}{2}\|\mathbf{w}\|^2 + \sum_{i=1}^n \xi_i \log\left(1 + e^{-z_i(\mathbf{w}^T\mathbf{x}_i+b)}\right), \quad \xi_i = \begin{cases}\frac{\pi_T}{n_T} & c_i=1 \\ \frac{1-\pi_T}{n_F} & c_i=0\end{cases}$$

Score → LLR: $s_{LLR} = s(\mathbf{x}) - \log\frac{\pi_T}{1-\pi_T}$ (use $\pi_T$, not $\pi_{emp}$)

In [ ]:
res_pw = []
for lam in lambdas:
    w, b = trainWeightedLogRegBinary(DTR, LTR, lam, pT=pT)
    sVal = (vrow(w) @ DVAL + b).ravel()
    sLLR = sVal - np.log(pT / (1 - pT))
    act = compute_actDCF(sLLR, LVAL, pT, 1.0, 1.0)
    mn  = compute_minDCF(sLLR, LVAL, pT, 1.0, 1.0)
    res_pw.append((lam, act, mn, sLLR))

display(pd.DataFrame([(f'{r[0]:.1e}', f'{r[1]:.4f}', f'{r[2]:.4f}') for r in res_pw],
                     columns=['λ', 'actDCF', 'minDCF']))

In [ ]:
fig, ax = plt.subplots(figsize=(9,5))
ax.plot(lambdas, [r[1] for r in res_pw], 'r-o', label='actDCF', linewidth=2)
ax.plot(lambdas, [r[2] for r in res_pw], 'b--o', label='minDCF', linewidth=2)
ax.set_xscale('log'); ax.set_xlabel('λ (log scale)'); ax.set_ylabel('DCF')
ax.set_title('Prior-Weighted LR (πT = 0.1) — Full Dataset')
ax.legend(); ax.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

### Answer: Are there significant differences? Are there advantages?

**No significant differences** compared to the standard model. Both minDCF and actDCF curves follow the same pattern. The prior-weighted model does not offer an advantage here.

**Why?** The empirical prior is $\pi_{emp} \approx 0.5$ (balanced dataset). With equal class sizes, re-weighting toward $\pi_T = 0.1$ multiplies class-0 samples by 5× relative to class-1 samples. In this dataset however, the linear boundary is dominated by the feature geometry, not the class imbalance — so the reweighting changes the boundary very little.

**When is prior-weighted LR useful?**
- When the training set has very different prior from the target application
- When you have very few samples from the target class and want to compensate

**Limitation:** Prior-weighted LR requires knowing $\pi_T$ at training time. If the deployment application changes, the model must be retrained.

## Part 4: Quadratic Logistic Regression (Full Dataset)

Map each 6-dimensional feature vector to a 42-dimensional vector:

$$\phi(\mathbf{x}) = \begin{bmatrix} \text{vec}(\mathbf{x}\mathbf{x}^T) \\ \mathbf{x} \end{bmatrix} \in \mathbb{R}^{42}$$

Then train standard LR on $\phi(\mathbf{x})$. This is equivalent to learning a **quadratic decision boundary** in the original 6D space.

In [ ]:
DTR_Q  = quadratic_expand(DTR)
DVAL_Q = quadratic_expand(DVAL)
print(f'Expanded feature dimension: {DTR_Q.shape[0]}')

res_quad = []
for lam in lambdas:
    w, b = trainLogRegBinary(DTR_Q, LTR, lam)
    sVal = (vrow(w) @ DVAL_Q + b).ravel()
    sLLR = sVal - np.log(pEmp / (1 - pEmp))
    act = compute_actDCF(sLLR, LVAL, pT, 1.0, 1.0)
    mn  = compute_minDCF(sLLR, LVAL, pT, 1.0, 1.0)
    res_quad.append((lam, act, mn, sLLR))

display(pd.DataFrame([(f'{r[0]:.1e}', f'{r[1]:.4f}', f'{r[2]:.4f}') for r in res_quad],
                     columns=['λ', 'actDCF', 'minDCF']))

In [ ]:
fig, ax = plt.subplots(figsize=(9,5))
ax.plot(lambdas, [r[1] for r in res_quad], 'r-o', label='actDCF', linewidth=2)
ax.plot(lambdas, [r[2] for r in res_quad], 'b--o', label='minDCF', linewidth=2)
ax.set_xscale('log'); ax.set_xlabel('λ (log scale)'); ax.set_ylabel('DCF')
ax.set_title('Quadratic LR — Full Dataset (πT = 0.1)')
ax.legend(); ax.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

### Answer: Is regularization effective? How does it affect the metrics?

**Yes — regularization is clearly effective for Quadratic LR**, unlike for linear LR:

- **Best minDCF (~0.244) at λ ≈ $3\times10^{-2}$** — there is a clear minimum. The expanded 42-dimensional space has many parameters that can overfit to noise, so regularization meaningfully controls the boundary complexity.
- **Low λ (e.g. $10^{-4}$):** Good minDCF (~0.260) but actDCF is poor due to calibration loss.
- **High λ ($\geq 1.0$):** minDCF degrades toward 0.326, losing the nonlinear advantage entirely.

**Why is quadratic LR so much better than linear LR (0.244 vs 0.361)?**

Features 5 & 6 have **different class-conditional variances** (multi-modal, different spreads). A linear boundary can only separate classes with different means. The quadratic expansion creates terms $x_i^2$ and $x_i x_j$ that allow the model to separate classes by their spreads, equivalent to fitting elliptical boundaries — the same information MVG uses via its covariance matrices.

**Best actDCF (~0.265 at λ = $3\times10^{-4}$):** Calibration is reasonable but not perfect at the best minDCF point. This is expected — the best discriminative model is not always the best-calibrated one.

## Part 5: Final Comparison — All Models (πT = 0.1)

Compare all trained models (logistic regression variants + Gaussian models from Lab 5) in terms of minimum DCF for the primary application.

In [ ]:
models_comp = {
    'Std LR\n(best λ)':  (min(r[2] for r in res_std),  'tab:blue'),
    'PW LR\n(best λ)':   (min(r[2] for r in res_pw),   'tab:green'),
    'Quad LR\n(best λ)': (min(r[2] for r in res_quad), 'tab:purple'),
    # These numbers are from lab 6
    'MVG':               (0.263, 'tab:orange'), 
    'Naive\nBayes':      (0.257, 'tab:cyan'),
    'Tied\nGaussian':    (0.363, 'tab:red'),
}
names = list(models_comp.keys())
vals  = [v[0] for v in models_comp.values()]
cols  = [v[1] for v in models_comp.values()]
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(names, vals, color=cols, edgecolor='black', alpha=0.85)
ax.bar_label(bars, fmt='%.3f', padding=3, fontsize=10, fontweight='bold')
ax.set_ylabel('minDCF (πT = 0.1)'); ax.set_ylim(0, 0.45)
ax.set_title('minDCF Comparison — All Models (Primary Application πT = 0.1)')
ax.grid(axis='y', alpha=0.3); plt.tight_layout(); plt.show()

### Answer: Which model achieves the best results? Why?

| Model | Best minDCF |
|---|---|
| **Quadratic LR** | **0.244** ← best |
| Naive Bayes | 0.257 |
| MVG | 0.263 |
| Standard LR | 0.361 |
| Prior-Weighted LR | 0.362 |
| Tied Gaussian | 0.363 |

**Best: Quadratic LR (minDCF = 0.244)**

**Separation rule / distribution assumption:**

Quadratic LR learns a quadratic decision boundary — it assigns a score $\mathbf{w}^T \phi(\mathbf{x}) + b$ where $\phi(\mathbf{x})$ includes all second-order terms. This is equivalent to a non-equal-covariance Gaussian model (like MVG), but trained discriminatively by minimizing cross-entropy rather than fitting densities.

**Why do Gaussian models and Quadratic LR beat linear models?**

As established in Lab 5, features 5 & 6 have **different class-conditional variances**. The optimal decision boundary for these features is a **curved (quadratic) boundary**, not a linear hyperplane. Linear models (standard LR, Tied Gaussian) cannot represent this boundary and are fundamentally limited to ~0.36 minDCF.

Quadratic LR beats MVG/Naive Bayes because:
- It is **discriminative** — directly optimizes the classification objective
- It does not waste capacity modeling the full density $p(\mathbf{x}|c)$
- It can discover nonlinear interactions ($x_i x_j$ terms) that MVG's diagonal/tied assumptions miss

**Save the Quadratic LR scores** — they will be needed for score calibration in future labs.